# MD-GRTN -- No Diffusion, Single Sequence (Recent Only)

Paper-aligned implementation of **MD-GRTN** (Bao et al., IEEE T-ITS 2025).

**Removed:** BackNet diffusion denoiser (MD module)

| Component | Paper spec | This notebook |
|---|---|---|
| Input | X_RecN only (recent 12 steps) | single recent seq |
| Diffusion (MD) | removed | removed |
| MAF | multi-head attn, Eq.5-9 | aligned |
| MGRC GCN+GRU layers | 6 | 6 |
| STFormer layers (L) | 3 | 3 |
| Spatial pos embed | A-weighted (Eq. 15) | yes |
| Temporal pos embed | hourly/daily/weekly (Eq. 21) | yes |
| d_model | 96 (heads=3) | 96 |
| Loss | Huber (Eq. 27) | yes |
| Optimizer | AdamW lr=0.001 wd=0.01 | yes |

In [ ]:
import os, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
    os.environ['PYTHONHASHSEED'] = str(seed)
    print(f'Seed set: {seed}')

set_seed()
print('PyTorch :', torch.__version__)
print('CUDA    :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU     :', torch.cuda.get_device_name(0))
    print('VRAM    :', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), 'GB')


In [ ]:
class Config:
    data_path    = '/content/PEMS08.npz'
    num_nodes    = 170          # PEMS08 nodes
    in_features  = 3            # flow, occupy, speed
    seq_len      = 12           # T_h  (input steps) -- paper Sec V-B-1
    pred_len     = 12           # T_f  (output steps)
    feature_idx  = 0            # predict flow channel

    # model -- paper Sec V-B-1
    d_model      = 96           # 96 / 3 = 32 per head
    n_heads      = 3
    num_layers   = 3            # L: STFormer layers (paper: 3)
    gru_layers   = 6            # MGRC GCN+GRU stacked layers (paper: 6)
    dropout      = 0.1

    # training -- paper Sec V-B-1
    batch_size   = 64
    lr           = 1e-3
    weight_decay = 0.01
    epochs       = 800          # paper: max 800 iterations
    patience     = 50
    delta_h      = 1.0          # Huber delta (Eq. 27)
    train_ratio  = 0.7
    val_ratio    = 0.1

    ckpt_path = 'mdgrtn_nodiff_ckpt.pt'
    best_path = 'mdgrtn_nodiff_best.pt'

cfg    = Config()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert cfg.d_model % cfg.n_heads == 0, 'd_model must be divisible by n_heads'
print(f'Config | d_model={cfg.d_model} heads={cfg.n_heads} head_dim={cfg.d_model//cfg.n_heads}')
print(f'num_layers(ST)={cfg.num_layers} | gru_layers(MGRC)={cfg.gru_layers} | device={device}')


In [ ]:
def load_pems(cfg):
    # Load PEMS npz, z-score normalise (paper Sec V-A), extract adjacency
    raw  = np.load(cfg.data_path)
    print('Keys:', list(raw.keys()))
    data = raw['data'].astype(np.float32)
    T, N, F = data.shape
    print(f'Shape: {data.shape}')

    # Per-sensor z-score normalisation (paper Sec V-A)
    mean = data.mean(axis=0)        # (N, F)
    std  = data.std(axis=0) + 1e-8
    data_norm = (data - mean[None]) / std[None]

    # Distance adjacency from dataset file (Eq. 11 source)
    for key in ('adjacency_matrix', 'adj_mx', 'adj'):
        if key in raw:
            A_dist = raw[key].astype(np.float32)
            print(f'Adjacency from "{key}" shape={A_dist.shape}')
            break
    else:
        print('No adjacency -- using identity')
        A_dist = np.eye(N, dtype=np.float32)

    # Row-normalise
    deg    = A_dist.sum(axis=1, keepdims=True) + 1e-8
    A_dist = A_dist / deg
    return data_norm, mean, std, A_dist


class TrafficDataset(Dataset):
    # Single-sequence dataset -- recent window only (no hour/day copies)
    def __init__(self, data, cfg, start, end):
        self.data = torch.from_numpy(data)
        self.S    = cfg.seq_len
        self.P    = cfg.pred_len
        self.fi   = cfg.feature_idx
        self.idx  = list(range(start + cfg.seq_len, end - cfg.pred_len + 1))

    def __len__(self): return len(self.idx)

    def __getitem__(self, i):
        t = self.idx[i]
        x = self.data[t - self.S : t]              # (S, N, F)  recent input
        y = self.data[t : t + self.P, :, self.fi]  # (P, N)     future flow
        return x, y


def build_dataloaders(cfg):
    set_seed()
    data, mean, std, A_dist = load_pems(cfg)
    T  = len(data)
    t1 = int(T * cfg.train_ratio)
    t2 = int(T * (cfg.train_ratio + cfg.val_ratio))

    kw = dict(batch_size=cfg.batch_size, num_workers=2, pin_memory=True)
    dl_tr = DataLoader(TrafficDataset(data, cfg, 0,  t1), shuffle=True,  **kw)
    dl_va = DataLoader(TrafficDataset(data, cfg, t1, t2), shuffle=False, **kw)
    dl_te = DataLoader(TrafficDataset(data, cfg, t2, T),  shuffle=False, **kw)
    print(f'Train={len(dl_tr.dataset)} | Val={len(dl_va.dataset)} | Test={len(dl_te.dataset)}')
    return dl_tr, dl_va, dl_te, mean, std, A_dist

print('Data utilities ready.')


In [ ]:
# ================================================================
# MAF MODULE  (paper Sec IV-A, Eq. 5-9, no diffusion)
#
# Without diffusion the raw (noisy) input feeds directly into Q/K/V.
# For a single sequence (n_seqs=1) the module applies multi-head
# self-attention over the node dimension for each time step.
#
#   Q_k = W^Q_k * X_k      (Eq. 5)
#   K_k = W^K_k * X_k      (Eq. 6)
#   V_k = W^V_k * X_k      (Eq. 7)
#   Att_k = softmax(QK^T / sqrt(d_j)) V   (Eq. 8)
#   X_F1 = Concat(heads) W_MH             (Eq. 9)
# ================================================================

class MAFModule(nn.Module):
    # Multi-head Attention Fusion -- paper Eq. 5-9
    # n_seqs=1: recent sequence only, diffusion removed
    # Each sequence has its own W_Q/W_K/W_V (paper: per-cycle matrices)
    def __init__(self, in_dim, d_model, n_heads, n_seqs=1):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_j     = d_model // n_heads   # per-head dimension (Eq. 8 scale)
        self.n_seqs  = n_seqs

        # Per-sequence learnable projection matrices (paper W^Q_k, W^K_k, W^V_k)
        self.W_Q = nn.ModuleList([nn.Linear(in_dim, d_model, bias=False) for _ in range(n_seqs)])
        self.W_K = nn.ModuleList([nn.Linear(in_dim, d_model, bias=False) for _ in range(n_seqs)])
        self.W_V = nn.ModuleList([nn.Linear(in_dim, d_model, bias=False) for _ in range(n_seqs)])

        # Output transform W_MH (paper Eq. 9)
        self.W_MH = nn.Linear(d_model * n_seqs, d_model)
        self.norm  = nn.LayerNorm(d_model)

    def forward(self, seqs):
        # seqs: list of n_seqs tensors, each (B, S, N, F)
        # returns X_F1 : (B, S, N, d_model)
        B, S, N, _ = seqs[0].shape
        head_outputs = []

        for k, x in enumerate(seqs):
            f = x.reshape(B * S, N, -1)           # (B*S, N, F)

            Q = self.W_Q[k](f)                     # (B*S, N, d_model)   Eq. 5
            K = self.W_K[k](f)                     # (B*S, N, d_model)   Eq. 6
            V = self.W_V[k](f)                     # (B*S, N, d_model)   Eq. 7

            # Scaled dot-product attention over nodes (Eq. 8)
            scores = torch.bmm(Q, K.transpose(1, 2)) / (self.d_j ** 0.5)
            attn   = F.softmax(scores, dim=-1)     # (B*S, N, N)
            head_k = torch.bmm(attn, V)            # (B*S, N, d_model)
            head_outputs.append(head_k.reshape(B, S, N, -1))

        # Concatenate all period heads and project  (Eq. 9)
        X_F1 = self.W_MH(torch.cat(head_outputs, dim=-1))
        return self.norm(X_F1)                     # (B, S, N, d_model)

print('MAF defined -- paper Eq. 5-9, single sequence, no diffusion.')


In [ ]:
# ================================================================
# MGRC MODULE  (paper Sec IV-B, Eq. 10-14)
#
#   A_dyna = softmax(relu(E1 E2^T))               (Eq. 10)
#   A_dist(i,j) = exp(-dist(vi,vj)^2/sigma^2)     (Eq. 11) from data
#   A_F = relu(Conv2D(concat(A_dyna, A_dist)))     (Eq. 12)
#   X'  = relu(A_F X W_GCN)                       (Eq. 13)
#   GRU: z_t, r_t, h_tilde, h_t                  (Eq. 14)
# ================================================================

class MultiGraphFusion(nn.Module):
    # Fuse dynamic + distance adjacency -- paper Eq. 10-12
    def __init__(self, N):
        super().__init__()
        self.E1     = nn.Parameter(torch.randn(N, 1) * 0.01)  # node embeddings Eq. 10
        self.E2     = nn.Parameter(torch.randn(N, 1) * 0.01)
        self.conv2d = nn.Conv2d(2, 1, kernel_size=1)           # fusion conv Eq. 12

    def forward(self, A_dist):
        # Dynamic adjacency  (Eq. 10)
        A_dyna = F.softmax(F.relu(self.E1 @ self.E2.T), dim=-1)    # (N, N)
        # Stack for Conv2D  (1, 2, N, N)
        stack  = torch.stack([A_dyna, A_dist], dim=0).unsqueeze(0)
        A_F    = F.relu(self.conv2d(stack).squeeze(0).squeeze(0))   # (N, N) Eq. 12
        return A_F / (A_F.sum(-1, keepdim=True) + 1e-8)


class GCN_GRU_Layer(nn.Module):
    # Single GCN + GRU recurrent layer -- paper Eq. 13-14
    def __init__(self, in_dim, hid_dim):
        super().__init__()
        self.hid_dim = hid_dim
        self.W_GCN   = nn.Linear(in_dim, hid_dim, bias=False)  # Eq. 13
        self.gru     = nn.GRUCell(hid_dim, hid_dim)             # Eq. 14

    def forward(self, x_seq, A_F):
        # x_seq: (B, S, N, in_dim)  |  A_F: (N, N)
        B, S, N, _ = x_seq.shape
        h = torch.zeros(B * N, self.hid_dim, device=x_seq.device)
        outs = []
        for t in range(S):
            agg = torch.einsum('nm,bmd->bnd', A_F, x_seq[:, t])  # graph conv (Eq. 13)
            xp  = F.relu(self.W_GCN(agg))                         # (B, N, hid)
            h   = self.gru(xp.reshape(B * N, -1), h)              # GRU (Eq. 14)
            outs.append(h.reshape(B, N, -1))
        return torch.stack(outs, dim=1)    # (B, S, N, hid_dim)


class MGRCModule(nn.Module):
    # Multi-Graph Recurrent Convolution -- paper Sec IV-B
    # Stacks n_layers=6 GCN+GRU layers (paper Sec V-B-1)
    def __init__(self, in_dim, hid_dim, N, n_layers=6):
        super().__init__()
        self.fusion = MultiGraphFusion(N)
        dims = [in_dim] + [hid_dim] * n_layers
        self.layers = nn.ModuleList([
            GCN_GRU_Layer(dims[i], dims[i + 1]) for i in range(n_layers)
        ])

    def forward(self, X_F1, A_dist):
        # X_F1: (B, S, N, d_model)  |  A_dist: (N, N)
        A_F = self.fusion(A_dist)
        x   = X_F1
        for layer in self.layers:
            x = layer(x, A_F)
        return x    # X_F2: (B, S, N, d_model)

print('MGRC defined -- 6 GCN+GRU layers, paper Eq. 10-14.')


In [ ]:
# ================================================================
# STFormer MODULE  (paper Sec IV-C, Eq. 15-26)
#
# Spatial Transformer layer (Eq. 15-19):
#   X^l_S1 = X^{l-1}_ST + A * W^l_SPE(X)    (Eq. 15) A-weighted SPE
#   X^l_S2 = SMHA(X^l_S1)                   (Eq. 16)
#   X^l_S3 = LN(X^l_S2 + X^l_S1)           (Eq. 17)
#   X^l_S4 = FC(X^l_S3)                     (Eq. 18)
#   Y^l_S  = LN(X^l_S4 + FC(X^l_S4))       (Eq. 19)
#
# Temporal Transformer layer (Eq. 20-25):
#   X^l_T1 = LN(X^{l-1}_ST + Y^l_S)        (Eq. 20)
#   X^l_T2 = X^l_T1 + W_h*E_hour
#                    + W_d*E_day
#                    + W_w*E_week           (Eq. 21)
#   X^l_T3 = TMHA(X^l_T2)                   (Eq. 22)
#   X^l_T3 = LN(X^l_T3 + X^l_T2)           (Eq. 23)
#   X^l_T4 = FC(X^l_T3)                     (Eq. 24)
#   X^l_ST = LN(X^l_T4 + FC(X^l_T4))       (Eq. 25)
# ================================================================

class SpatialTransformerLayer(nn.Module):
    # Spatial Transformer with adjacency-weighted SPE -- paper Eq. 15-19
    def __init__(self, d, n_heads, dropout):
        super().__init__()
        self.W_SPE = nn.Linear(d, d, bias=False)     # SPE weight (Eq. 15)
        self.attn  = nn.MultiheadAttention(d, n_heads, dropout=dropout, batch_first=True)
        self.ff1   = nn.Sequential(nn.Linear(d, d * 4), nn.ReLU(), nn.Linear(d * 4, d))
        self.ff2   = nn.Sequential(nn.Linear(d, d * 4), nn.ReLU(), nn.Linear(d * 4, d))
        self.norm1 = nn.LayerNorm(d)
        self.norm2 = nn.LayerNorm(d)
        self.drop  = nn.Dropout(dropout)

    def forward(self, x, A):
        # x: (B, S, N, d)  |  A: (N, N)
        B, S, N, d = x.shape

        # Eq. 15: X_S1 = x + A * W_SPE(x)   (A aggregates neighbourhood SPE)
        spe   = self.W_SPE(x)                                  # (B, S, N, d)
        spe_a = torch.einsum('nm,bsmd->bsnd', A, spe)         # A-weighted
        X_S1  = x + spe_a

        # Eq. 16: multi-head spatial attention over N nodes
        f      = X_S1.reshape(B * S, N, d)
        h, _   = self.attn(f, f, f)
        X_S2   = h.reshape(B, S, N, d)

        # Eq. 17: add & norm
        X_S3 = self.norm1(X_S2 + X_S1)

        # Eq. 18: feed-forward
        X_S4 = self.ff1(X_S3)

        # Eq. 19: second FF + add & norm
        Y_S  = self.norm2(X_S4 + self.ff2(X_S4))
        return Y_S    # (B, S, N, d)


class TemporalTransformerLayer(nn.Module):
    # Temporal Transformer with hourly/daily/weekly PE -- paper Eq. 20-25
    def __init__(self, d, n_heads, dropout, seq_len):
        super().__init__()
        self.seq_len = seq_len

        # Temporal position encodings (Eq. 21): hour in [1,60], day in [1,24], week in [1,7]
        self.E_hour = nn.Embedding(61, d)
        self.E_day  = nn.Embedding(25, d)
        self.E_week = nn.Embedding(8,  d)

        # Per-node temporal weighting matrices W_hour, W_day, W_week (Eq. 21)
        self.W_hour = nn.Linear(d, d, bias=False)
        self.W_day  = nn.Linear(d, d, bias=False)
        self.W_week = nn.Linear(d, d, bias=False)

        self.attn  = nn.MultiheadAttention(d, n_heads, dropout=dropout, batch_first=True)
        self.ff1   = nn.Sequential(nn.Linear(d, d * 4), nn.ReLU(), nn.Linear(d * 4, d))
        self.ff2   = nn.Sequential(nn.Linear(d, d * 4), nn.ReLU(), nn.Linear(d * 4, d))
        self.norm1 = nn.LayerNorm(d)
        self.norm2 = nn.LayerNorm(d)
        self.drop  = nn.Dropout(dropout)

    def _time_pe(self, S, device):
        # Cyclic time indices from sequence position
        # In production use actual timestamps; here approximate with modulo
        steps    = torch.arange(S, device=device)
        idx_hour = (steps % 60 + 1).long()   # [1,60]
        idx_day  = (steps % 24 + 1).long()   # [1,24]
        idx_week = (steps % 7  + 1).long()   # [1,7]
        return idx_hour, idx_day, idx_week

    def forward(self, x_prev, Y_S):
        # x_prev: (B,S,N,d) X^{l-1}_ST  |  Y_S: (B,S,N,d) spatial output
        B, S, N, d = x_prev.shape
        device = x_prev.device

        # Eq. 20: add & norm with spatial transformer output
        X_T1 = self.norm1(x_prev + Y_S)

        # Eq. 21: temporal position encoding (hourly + daily + weekly)
        idx_h, idx_d, idx_w = self._time_pe(S, device)
        pe = (self.W_hour(self.E_hour(idx_h))
            + self.W_day(self.E_day(idx_d))
            + self.W_week(self.E_week(idx_w)))        # (S, d)
        X_T2 = X_T1 + pe.unsqueeze(0).unsqueeze(2)   # broadcast (B,S,N,d)

        # Eq. 22: temporal multi-head attention over S time steps
        f    = X_T2.permute(0, 2, 1, 3).reshape(B * N, S, d)
        h, _ = self.attn(f, f, f)
        h    = h.reshape(B, N, S, d).permute(0, 2, 1, 3)

        # Eq. 23: add & norm
        X_T3 = self.norm1(h + X_T2)

        # Eq. 24: feed-forward
        X_T4 = self.ff1(X_T3)

        # Eq. 25: second FF + add & norm
        X_ST = self.norm2(X_T4 + self.ff2(X_T4))
        return X_ST    # (B, S, N, d)

print('SpatialTransformerLayer + TemporalTransformerLayer defined -- paper Eq. 15-25.')


In [ ]:
class MDGRTN_NoDiff(nn.Module):
    # MD-GRTN without diffusion -- paper-aligned (Bao et al. IEEE T-ITS 2025)
    #
    # Pipeline (Fig. 5):
    #   x_rec -> MAF (Eq.5-9)
    #         -> MGRC (Eq.10-14)
    #         -> [Spatial Transformer (Eq.15-19)
    #             Temporal Transformer (Eq.20-25)] x L
    #         -> FC output (Eq.26)
    #
    # Differences from full MD-GRTN:
    #   - No BackNet diffusion (MD module removed)
    #   - Single input sequence X_RecN only (no hourly/daily copies)
    def __init__(self, cfg):
        super().__init__()
        N, F, d  = cfg.num_nodes, cfg.in_features, cfg.d_model
        L, h, dr = cfg.num_layers, cfg.n_heads, cfg.dropout
        S, P     = cfg.seq_len, cfg.pred_len

        # MAF (Sec IV-A, Eq. 5-9): n_seqs=1, no diffusion
        self.maf  = MAFModule(F, d, h, n_seqs=1)

        # MGRC (Sec IV-B, Eq. 10-14): 6 stacked GCN+GRU layers
        self.mgrc = MGRCModule(d, d, N, n_layers=cfg.gru_layers)

        # STFormer (Sec IV-C, Eq. 15-25): L paired spatial+temporal layers
        self.spatial_layers  = nn.ModuleList([
            SpatialTransformerLayer(d, h, dr) for _ in range(L)])
        self.temporal_layers = nn.ModuleList([
            TemporalTransformerLayer(d, h, dr, S) for _ in range(L)])

        # Output FC (Eq. 26): Y_Feat = FC(Y_ST)
        # Applied on last time step to produce P predictions per node
        self.out_fc = nn.Sequential(
            nn.Linear(d, d),
            nn.ReLU(),
            nn.Linear(d, P)
        )

    def forward(self, x, A_dist):
        # x      : (B, S, N, F)  recent noisy input sequence
        # A_dist : (N, N)         row-normalised distance adjacency
        # returns: (B, P, N)      predicted future flow

        # Step 1 -- MAF: attention fusion over nodes (no BackNet)
        X_F1 = self.maf([x])              # (B, S, N, d)

        # Step 2 -- MGRC: dual-graph GCN+GRU
        X_F2 = self.mgrc(X_F1, A_dist)   # (B, S, N, d)

        # Step 3 -- STFormer: L pairs of spatial + temporal transformer layers
        X_ST = X_F2
        for sp_layer, tmp_layer in zip(self.spatial_layers, self.temporal_layers):
            Y_S  = sp_layer(X_ST, A_dist)   # Eq. 15-19
            X_ST = tmp_layer(X_ST, Y_S)     # Eq. 20-25

        # Step 4 -- Output (Eq. 26): FC on last time step
        Y_last  = X_ST[:, -1, :, :]    # (B, N, d)
        Y_Feat  = self.out_fc(Y_last)  # (B, N, P)
        return Y_Feat.permute(0, 2, 1) # (B, P, N)


set_seed()
model = MDGRTN_NoDiff(cfg).to(device)
total = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model ready | Parameters: {total:,}')
print(f'  No diffusion | MAF(1 seq) -> MGRC(x{cfg.gru_layers}) -> STFormer(L={cfg.num_layers})')

# Sanity-check forward pass
with torch.no_grad():
    dummy = torch.randn(2, cfg.seq_len, cfg.num_nodes, cfg.in_features).to(device)
    adj   = torch.eye(cfg.num_nodes).to(device)
    out   = model(dummy, adj)
print(f'Output shape: {tuple(out.shape)}  (expected [2, {cfg.pred_len}, {cfg.num_nodes}])')


In [ ]:
def masked_mae(pred, true):
    return torch.abs(pred - true).mean()

def masked_rmse(pred, true):
    return ((pred - true) ** 2).mean().sqrt()

def masked_mape(pred, true, low_thresh=10.0):
    # Exclude near-zero ground-truth to avoid instability
    mask = (true.abs() > low_thresh).float()
    if mask.sum() < 1:
        return torch.tensor(0.0, device=pred.device)
    return (torch.abs((pred - true) / (true.abs() + 1.0)) * mask).sum() / mask.sum() * 100

def huber_loss(pred, true, delta=1.0):
    # Paper Eq. 27
    return F.huber_loss(pred, true, delta=delta)

print('Metrics defined (MAE, RMSE, MAPE, Huber -- Eq. 27).')


In [ ]:
# Uncomment to mount Google Drive:
# from google.colab import drive
# drive.mount('/content/drive')
# cfg.data_path = '/content/drive/MyDrive/PEMS08.npz'

dl_train, dl_val, dl_test, mean_np, std_np, A_dist_np = build_dataloaders(cfg)

mean_flow = torch.from_numpy(mean_np[:, cfg.feature_idx]).to(device)  # (N,)
std_flow  = torch.from_numpy(std_np[:,  cfg.feature_idx]).to(device)
A_dist    = torch.from_numpy(A_dist_np).to(device)                    # (N, N)

print(f'mean_flow: min={mean_flow.min():.2f}  max={mean_flow.max():.2f}')
print(f'std_flow:  min={std_flow.min():.4f}   max={std_flow.max():.2f}')
print(f'Train: {len(dl_train)} batches | Val: {len(dl_val)} | Test: {len(dl_test)}')


In [ ]:
def train_epoch(model, loader, optimizer, A_dist, device):
    model.train()
    total = 0.0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        pred = model(x, A_dist)
        loss = huber_loss(pred, y, cfg.delta_h)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        total += loss.item()
    return total / len(loader)


@torch.no_grad()
def eval_epoch(model, loader, A_dist, device, mean_flow, std_flow):
    model.eval()
    maes, rmses, mapes = [], [], []
    for x, y in loader:
        x, y   = x.to(device), y.to(device)
        pred   = model(x, A_dist)
        # Inverse z-score
        pred_d = pred * std_flow[None, None, :] + mean_flow[None, None, :]
        y_d    = y    * std_flow[None, None, :] + mean_flow[None, None, :]
        maes.append(masked_mae(pred_d, y_d).item())
        rmses.append(masked_rmse(pred_d, y_d).item())
        mapes.append(masked_mape(pred_d, y_d).item())
    return np.mean(maes), np.mean(rmses), np.mean(mapes)

print('Train / eval functions defined.')


In [ ]:
def save_checkpoint(model, optimizer, scheduler, epoch, best_mae, history, cfg, drive_path=None):
    torch.save({
        'model_state':     model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'scheduler_state': scheduler.state_dict(),
        'epoch':           epoch,
        'best_mae':        best_mae,
        'history':         history,
        'seed':            SEED,
    }, cfg.ckpt_path)
    if drive_path:
        import shutil; shutil.copy(cfg.ckpt_path, drive_path)
    print(f'Ckpt saved ep={epoch} | best_mae={best_mae:.3f}', end='\r')


def load_checkpoint(model, optimizer, scheduler, cfg, drive_path=None):
    if drive_path:
        import shutil; shutil.copy(drive_path, cfg.ckpt_path)
    if not os.path.exists(cfg.ckpt_path):
        print('No checkpoint -- starting fresh.')
        return 1, float('inf'), {'train_loss': [], 'val_mae': [], 'val_rmse': [], 'val_mape': []}
    ckpt = torch.load(cfg.ckpt_path, map_location=device)
    model.load_state_dict(ckpt['model_state'])
    optimizer.load_state_dict(ckpt['optimizer_state'])
    scheduler.load_state_dict(ckpt['scheduler_state'])
    print(f'Resumed from epoch {ckpt["epoch"]} | best_mae={ckpt["best_mae"]:.3f}')
    return ckpt['epoch'] + 1, ckpt['best_mae'], ckpt['history']

print('Checkpoint utilities ready.')


In [ ]:
set_seed()

optimizer = torch.optim.AdamW(
    model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
# CosineAnnealingLR matches paper Fig. 7 convergence (800 iterations)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=cfg.epochs, eta_min=1e-6)

# To RESUME from checkpoint uncomment:
# start_epoch, best_val_mae, history = load_checkpoint(
#     model, optimizer, scheduler, cfg,
#     drive_path='/content/drive/MyDrive/mdgrtn_nodiff_ckpt.pt')

DRIVE_CKPT   = None   # set Drive path string to auto-save each epoch
start_epoch  = 1
best_val_mae = float('inf')
patience_cnt = 0
history      = {'train_loss': [], 'val_mae': [], 'val_rmse': [], 'val_mape': []}

print(f'Training | max_epochs={cfg.epochs} | patience={cfg.patience}')
print(f'Model    | No diffusion | MAF(1-seq) -> MGRC(x{cfg.gru_layers}) -> STFormer(L={cfg.num_layers})')
print(f'Baseline | MAE=13.114  RMSE=22.623  MAPE=8.471%  (PEMS08, Table V)')
print('=' * 68)

for epoch in range(start_epoch, cfg.epochs + 1):
    train_loss = train_epoch(model, dl_train, optimizer, A_dist, device)
    val_mae, val_rmse, val_mape = eval_epoch(
        model, dl_val, A_dist, device, mean_flow, std_flow)
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['val_mae'].append(val_mae)
    history['val_rmse'].append(val_rmse)
    history['val_mape'].append(val_mape)

    tag = ''
    if val_mae < best_val_mae:
        best_val_mae = val_mae
        patience_cnt = 0
        torch.save(model.state_dict(), cfg.best_path)
        tag = '  <- best'
    else:
        patience_cnt += 1
        if patience_cnt >= cfg.patience:
            print(f'Early stopping at epoch {epoch}.')
            break

    print(f'Ep {epoch:04d} | Loss={train_loss:.4f} | '
          f'MAE={val_mae:.3f}  RMSE={val_rmse:.3f}  MAPE={val_mape:.2f}%{tag}')

    save_checkpoint(model, optimizer, scheduler, epoch,
                    best_val_mae, history, cfg, drive_path=DRIVE_CKPT)

print(f'\nBest Val MAE: {best_val_mae:.3f}')


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(history['train_loss'], color='steelblue', lw=2)
axes[0].set_title('Huber Loss (train)'); axes[0].set_xlabel('Epoch')

axes[1].plot(history['val_mae'], color='orange', lw=2, label='Val MAE')
axes[1].axhline(13.114, color='red', ls='--', lw=1.5, label='Baseline 13.114')
axes[1].set_title('Validation MAE'); axes[1].legend()

axes[2].plot(history['val_rmse'], color='green', lw=2, label='Val RMSE')
axes[2].axhline(22.623, color='red', ls='--', lw=1.5, label='Baseline 22.623')
axes[2].set_title('Validation RMSE'); axes[2].legend()

plt.tight_layout()
plt.savefig('curves_nodiff.png', dpi=150)
plt.show()
print('Training curves saved -> curves_nodiff.png')


In [ ]:
model.load_state_dict(torch.load(cfg.best_path, map_location=device))

@torch.no_grad()
def paper_eval(model, loader, A_dist, device, mean_flow, std_flow):
    model.eval()
    all_pred, all_true = [], []
    for x, y in loader:
        x, y   = x.to(device), y.to(device)
        pred   = model(x, A_dist)
        pred_d = pred * std_flow[None, None, :] + mean_flow[None, None, :]
        y_d    = y    * std_flow[None, None, :] + mean_flow[None, None, :]
        all_pred.append(pred_d.cpu())
        all_true.append(y_d.cpu())

    P = torch.cat(all_pred, dim=0)
    Y = torch.cat(all_true, dim=0)

    mae  = torch.abs(P - Y).mean().item()
    rmse = ((P - Y) ** 2).mean().sqrt().item()
    mask = Y.abs() > 10.0
    mape = (torch.abs((P[mask] - Y[mask]) / (Y[mask].abs() + 1.0))).mean().item() * 100

    print('\n' + '=' * 60)
    print('  TEST RESULTS -- MD-GRTN (no diffusion, single sequence)')
    print('=' * 60)
    print(f'  MAE  : {mae:.3f}    baseline (Table V): 13.114   D={mae-13.114:+.3f}')
    print(f'  RMSE : {rmse:.3f}    baseline (Table V): 22.623   D={rmse-22.623:+.3f}')
    print(f'  MAPE : {mape:.3f}%   baseline (Table V):  8.471%  D={mape-8.471:+.3f}%')
    print('=' * 60)
    return mae, rmse, mape

paper_eval(model, dl_test, A_dist, device, mean_flow, std_flow)


In [ ]:
@torch.no_grad()
def horizon_eval(model, loader, A_dist, device, mean_flow, std_flow):
    # Per-horizon evaluation at 15/30/60 min -- paper Fig. 13
    model.eval()
    buf = {2: {'mae':[],'rmse':[],'mape':[]},
           5: {'mae':[],'rmse':[],'mape':[]},
           11:{'mae':[],'rmse':[],'mape':[]}}
    for x, y in loader:
        x, y   = x.to(device), y.to(device)
        pred   = model(x, A_dist)
        pred_d = pred * std_flow[None, None, :] + mean_flow[None, None, :]
        y_d    = y    * std_flow[None, None, :] + mean_flow[None, None, :]
        for h in buf:
            buf[h]['mae'].append(masked_mae(pred_d[:, h, :], y_d[:, h, :]).item())
            buf[h]['rmse'].append(masked_rmse(pred_d[:, h, :], y_d[:, h, :]).item())
            buf[h]['mape'].append(masked_mape(pred_d[:, h, :], y_d[:, h, :]).item())

    label = {2: '3-step  (15 min)', 5: '6-step  (30 min)', 11: '12-step (60 min)'}
    print(f'  {"Horizon":>17} | {"MAE":>8} | {"RMSE":>8} | {"MAPE":>9}')
    print('-' * 55)
    for h in [2, 5, 11]:
        m = {k: np.mean(v) for k, v in buf[h].items()}
        print(f'  {label[h]:>17} | {m["mae"]:>8.3f} | {m["rmse"]:>8.3f} | {m["mape"]:>8.2f}%')

horizon_eval(model, dl_test, A_dist, device, mean_flow, std_flow)
